# Unit 5 - Exercise 1: Model Selection Practice

**The data:** the real 1912 Titanic passenger manifest, loaded with `load("titanic")` —
891 people, predicting who survived. The loading cell
below handles the two real gaps in the file (177 missing ages, 2 missing ports) and
encodes the categorical columns, so you can spend this exercise on tuning rather than on
cleaning.

Keep one number in mind while you tune: **61.6% of the passengers died**, so any model
scoring below 0.616 is worse than a model that always guesses "died".

# Instructions:
# 1. Load the provided dataset (done for you in the cell below)
# 2. Use Grid Search to find best hyperparameters for Random Forest
# 3. Compare with default Random Forest
# 4. (Optional) Try XGBoost if available
# 5. Evaluate and compare results
# 6. Measure the error per group for the default and the tuned model

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- `load("titanic")` — the real passenger manifest, all 891 rows. The whole file ships with the
  repository and the loader finds it from any working directory, Google Colab included, so there
  is no file path to get wrong

**Outputs:** What you'll see when you run the cells

- A default Random Forest's score, the best hyperparameters found by grid search, an
  honest comparison of the two on held-out passengers, and the error of each per group.

---

In [1]:
# --- Data setup. Works from any folder, and on Google Colab. -------------------------
# WHAT: find the repository root, put it on sys.path, then import the shared data loader.
# WHY:  a hard-coded relative path such as ../../datasets/raw/<file>.csv only resolves when
#       the kernel's working directory happens to be this notebook's folder. Open it from the
#       repository root, from JupyterLab at another level, or on Colab where there is no
#       repository at all, and it breaks. load() finds the data wherever you are, downloads
#       it if this machine has none, and says out loud which copy it used.
import sys, pathlib

_here = pathlib.Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "tools" / "data.py").exists()), None)
if _root is None:                     # Google Colab, or a stray copy of the notebook
    import urllib.request
    pathlib.Path("tools").mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/A-Alwabel/"
            "AI-Diploma-Program/main/tools/data.py", "tools/data.py")
    except Exception as _e:
        raise RuntimeError(
            "Could not find the AI Diploma repository from this folder, and could not "
            "download the data loader either. Open this notebook inside a clone of "
            "https://github.com/A-Alwabel/AI-Diploma-Program, or connect to the internet "
            f"and re-run this cell. (underlying error: {_e})") from None
    _root = pathlib.Path.cwd()
sys.path.insert(0, str(_root))

from tools.data import load        # load("titanic"), load("creditcard_fraud", prefer="sample"), ...
# -------------------------------------------------------------------------------------
print("Data loader ready - no file paths needed, and this works on Colab too.")

Data loader ready - no file paths needed, and this works on Colab too.


In [2]:
# Import the tuning toolkit: GridSearchCV automates the try-every-combination
# loop over a Random Forest's hyperparameters.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

Load and prepare the real dataset

In [3]:
# Load the REAL Titanic manifest and turn it into a numeric feature matrix.
# WHY the fillna lines: 177 passengers have no recorded age and 2 have no port. Those are
# real gaps in a 1912 record, and a model cannot train until you decide what to do with them.
titanic = load("titanic")
print(f"Raw file: {titanic.shape[0]} passengers, {titanic.shape[1]} columns")
print("Missing values in the columns we use:")
print(titanic[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]
      .isna().sum().to_string())

clean = titanic.copy()
clean["Age"] = clean["Age"].fillna(clean["Age"].median())
clean["Embarked"] = clean["Embarked"].fillna(clean["Embarked"].mode()[0])

df = pd.DataFrame({
    "pclass": clean["Pclass"],
    "is_female": (clean["Sex"] == "female").astype(int),
    "age": clean["Age"],
    "sibsp": clean["SibSp"],
    "parch": clean["Parch"],
    "fare": clean["Fare"],
    "embarked_C": (clean["Embarked"] == "C").astype(int),
    "embarked_Q": (clean["Embarked"] == "Q").astype(int),
})
df["target"] = clean["Survived"]

X = df.drop(columns="target").to_numpy(dtype=float)
y = df["target"].to_numpy()

print("\nDataset ready!")
print(f"Shape: {df.shape}  (last column is the target)")
print(f"Class balance: {np.bincount(y)} (died, survived)")
print(f"Baseline to beat — always predict 'died': {(y == 0).mean():.4f}")

titanic: full file, 891 rows.
Raw file: 891 passengers, 12 columns
Missing values in the columns we use:
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2

Dataset ready!
Shape: (891, 9)  (last column is the target)
Class balance: [549 342] (died, survived)
Baseline to beat — always predict 'died': 0.6162


**How to work:** each task below is a `# Your code here...` scaffold. Write the code,
run the cell, and read the number you get before moving on.

## Task 1: Split the data

Use `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)` and print the
size and class balance of both halves. Why does `stratify=y` matter on a dataset that is
62/38 rather than 50/50?


In [4]:
print("\nTask 1: Split data")
# Your code here...


Task 1: Split data


## Task 2: Train default Random Forest

Fit `RandomForestClassifier(random_state=42)` with no tuning at all, then print its
training accuracy AND its test accuracy. The gap between them is the thing tuning is
supposed to close.


In [5]:
print("\nTask 2: Default Random Forest")
# Your code here...


Task 2: Default Random Forest


## Task 3: Grid Search for Random Forest

Run `GridSearchCV` over a small grid — for example `n_estimators` in [50, 100, 200],
`max_depth` in [3, 5, 10, None], `min_samples_leaf` in [1, 5, 10] — with `cv=5`. Print
`best_params_`, `best_score_` (the cross-validated score on the TRAINING data) and how
many fits it ran. Note how long it took.


In [6]:
print("\nTask 3: Grid Search")
# Your code here...


Task 3: Grid Search


## Task 4: Compare results

Score the tuned model on the held-out test set and put three numbers side by side: the
default forest's test accuracy, the tuned forest's test accuracy, and the always-predict-
"died" baseline. Then answer honestly in a comment: **was the tuning worth it?** On real
data the answer is often "a little, and less than the cross-validated score promised" —
if that is what you see, say so rather than rounding it up.


In [7]:
print("\nTask 4: Compare")
# Your code here...



Task 4: Compare


## Task 5: Did tuning move the error per group?

Grid search picked the forest with the best cross-validated accuracy and saw nothing else. Column 1
of `X` is `is_female` (74.2% of the women in this file survived, 18.9% of the men). For **both** the
default forest (Task 2) and the tuned forest (Task 4), split the test predictions by that column and
print per group: `n`, TPR, FPR and accuracy, plus the gap (max minus min) for each — Course 06
(AIAT 116) calls these *equalized odds* and *per-group accuracy*. Hint:
`confusion_matrix(y_true[mask], y_pred[mask], labels=[0, 1]).ravel()` returns `tn, fp, fn, tp`, and
`mask = X_test[:, 1] == 1` selects the women.

Then answer in a comment: did tuning narrow the gaps, widen them, or move them in different
directions? Whatever you find, the search could not have known — say in one line what you would add
to the selection step so that it could.


In [8]:
print("\nTask 5: Error per group, before and after tuning")
from sklearn.metrics import confusion_matrix
# Your code here...

print("\nExercise 1 Complete!")


Task 5: Error per group, before and after tuning

Exercise 1 Complete!
